# Детоксификация русскоязычного текста

**Задача:** переписать токсичную реплику в нейтральном стиле, сохранив основной смысл.

**Стек:** Transformers, Detoxify, Sentence Transformers, PyTorch, pandas.

Качество оценивается по двум направлениям: снижению токсичности и семантической близости исходного и преобразованного текста.


> Датасет содержит оскорбительные высказывания. Они используются только как тестовые данные и могут появиться в выводе демонстрационных ячеек.


In [ ]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from detoxify import Detoxify
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

SEED = 42
SAMPLE_SIZE = 100
DETOX_MODEL_NAME = "s-nlp/ruT5-base-detox"
TOKENIZER_NAME = "ai-forever/ruT5-base"
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
dataset = load_dataset("ilyiniv1755/toxic-russian-dataset", split="train")
data = pd.DataFrame(dataset)[["text", "label"]].dropna().drop_duplicates("text")
toxic = data[data["label"].astype(int) == 1]
evaluation_data = toxic.sample(
    n=min(SAMPLE_SIZE, len(toxic)), random_state=SEED
).reset_index(drop=True)

detox_tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
detox_model = AutoModelForSeq2SeqLM.from_pretrained(DETOX_MODEL_NAME).to(DEVICE)
detox_model.eval()

toxicity_model = Detoxify("multilingual", device=str(DEVICE))
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=str(DEVICE))

print(f"Примеров для оценки: {len(evaluation_data)}; устройство: {DEVICE}")


In [ ]:
def detoxify_text(text: str) -> str:
    encoded = detox_tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        max_length=256,
    ).to(DEVICE)

    with torch.inference_mode():
        generated = detox_model.generate(
            **encoded,
            num_beams=5,
            max_new_tokens=256,
            no_repeat_ngram_size=3,
            repetition_penalty=1.15,
            early_stopping=True,
        )
    result = detox_tokenizer.decode(generated[0], skip_special_tokens=True).strip()
    return result or str(text)


def toxicity_scores(texts: list[str]) -> np.ndarray:
    prediction = toxicity_model.predict(texts)
    key = "toxicity" if "toxicity" in prediction else "toxic"
    return np.asarray(prediction[key], dtype=float)


In [ ]:
original_texts = evaluation_data["text"].astype(str).tolist()
detoxified_texts = [detoxify_text(text) for text in original_texts]

original_toxicity = toxicity_scores(original_texts)
detoxified_toxicity = toxicity_scores(detoxified_texts)

original_embeddings = embedding_model.encode(
    original_texts, convert_to_tensor=True, show_progress_bar=True
)
detoxified_embeddings = embedding_model.encode(
    detoxified_texts, convert_to_tensor=True, show_progress_bar=True
)
semantic_similarity = np.array(
    [
        cos_sim(original_embeddings[i], detoxified_embeddings[i]).item()
        for i in range(len(original_texts))
    ]
)

results = pd.DataFrame(
    {
        "original": original_texts,
        "detoxified": detoxified_texts,
        "toxicity_before": original_toxicity,
        "toxicity_after": detoxified_toxicity,
        "toxicity_reduction": original_toxicity - detoxified_toxicity,
        "semantic_similarity": semantic_similarity,
    }
)


In [ ]:
summary = pd.Series(
    {
        "Mean toxicity before": results["toxicity_before"].mean(),
        "Mean toxicity after": results["toxicity_after"].mean(),
        "Mean reduction": results["toxicity_reduction"].mean(),
        "Share with reduction > 0.10": (results["toxicity_reduction"] > 0.10).mean(),
        "Mean semantic similarity": results["semantic_similarity"].mean(),
    }
)
summary.round(4).to_frame("value")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(
    data=results,
    x="toxicity_before",
    y="toxicity_after",
    ax=axes[0],
)
axes[0].plot([0, 1], [0, 1], "--", color="gray")
axes[0].set_title("Токсичность до и после")

sns.histplot(results["toxicity_reduction"], bins=15, ax=axes[1])
axes[1].axvline(0, linestyle="--", color="gray")
axes[1].set_title("Распределение снижения токсичности")
plt.tight_layout()


In [ ]:
pd.set_option("display.max_colwidth", 120)
results.sort_values("toxicity_reduction", ascending=False).head(10)


## Ограничения

Автоматический детектор может наследовать смещения обучающих данных. Высокое снижение токсичности само по себе не гарантирует сохранение фактов и намерения автора, поэтому для чувствительных сценариев нужен ручной контроль и отдельная проверка качества на целевом домене.
